# NASDAQ-100 data-stage demo

Owner: Data (Chotima), issues #34/#60. This notebook runs the implemented data stages without
network or artifact writes. Default data are synthetic fixtures, not empirical results.
Training, checkpoint inference and attention tracing remain pending the model team's code.


In [ ]:
import os
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
repo = next((p for p in candidates if (p / "SCHEMA.md").exists() and (p / "src").is_dir()), None)
if repo is None:
    raise RuntimeError("Open the notebook inside the nasdaq100-news-graph checkout")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
os.environ.setdefault("NDX_DATA_DIR", str(repo / "tests/fixtures"))
from src.core.paths import data_dir, FIXTURES_DIR
synthetic = data_dir().resolve() == FIXTURES_DIR.resolve()
print("Data:", data_dir(), "SYNTHETIC" if synthetic else "USER DATA")


In [ ]:
from src.core.io import read_table
from src.data.fetch_prices import add_in_index
from src.data.build_labels import make_labels
from src.data.indicators import compute_indicators, FEATURES
from src.data.quality import quality_report
from src.data.coverage import news_coverage

membership = read_table("membership")
prices = read_table("prices")
stored_labels = read_table("labels")
from src.core.paths import table_path
news = read_table("news_features") if table_path("news_features").exists() else None
articles = read_table("article_tickers") if table_path("article_tickers").exists() else None
print({"membership": membership.shape, "prices": prices.shape, "labels": stored_labels.shape,
       "news_features": None if news is None else news.shape,
       "article_tickers": None if articles is None else articles.shape})


## Membership, labels and quality

Warm-up/out-of-index rows stay available for features; the model sample filters `in_index`.
Labels use actual NASDAQ sessions. Old synthetic fixture labels use generic business-day
row shifts, so the quality report can flag calendar differences. Nothing is overwritten.


In [ ]:
import pandas as pd
checked = add_in_index(prices.drop(columns="in_index"), membership)
pd.testing.assert_series_equal(checked.in_index, prices.in_index)
labels = make_labels(prices)
features = compute_indicators(prices)
quality = quality_report(prices, stored_labels, membership, start="2019-01-01", end="2023-12-31")
coverage = news_coverage(prices, articles) if articles is not None else None
print("In-index:", int(prices.in_index.sum()), "Warm-up/out-of-index:", int((~prices.in_index).sum()))
print(quality.groupby("check").size().to_string())
print(coverage.head().to_string(index=False) if coverage is not None else "News coverage pending NLP artifact")


In [ ]:
feature_view = features.rename(columns={"ret_1d": "past_ret_1d"})
sample = prices[["date", "ticker", "in_index"]].merge(
    feature_view, on=["date", "ticker"], validate="one_to_one"
).merge(labels, on=["date", "ticker"], validate="one_to_one")
if news is not None:
    sample = sample.merge(news[["date", "ticker", "has_news", "n_articles", "sentiment"]],
                          on=["date", "ticker"], how="left", validate="one_to_one")
else:
    print("News features pending; absent upstream data are not encoded as neutral/no-news")
feature_columns = ["past_ret_1d", *FEATURES[1:]]
ready = sample.loc[sample.in_index & sample.date.between("2019-01-01", "2023-12-31")].dropna(
    subset=[*feature_columns, "y_1d", "y_5d"]
)
print("Price-feature-ready rows (data completeness still requires review):", len(ready))
preview_columns = ["date", "ticker", "past_ret_1d", "rsi_14", "y_1d", "y_5d"]
if news is not None:
    preview_columns += ["has_news"]
print(ready[preview_columns].head().to_string(index=False))
# Only feature_columns go into the price encoder. Forward returns/labels are targets.
# Fit a model scaler on a training fold, never on all rows in ready.


## Saved predictions and remaining integration

For fixtures, `predictions/random.parquet` is a schema example only. It is not trained
GAT/LSTM output. For real data this cell looks for `predictions/gat_lstm.parquet`.
Loading a saved artifact does not perform inference. Once Modelling provides a trained
checkpoint and inference interface, replace this preview with test-day inference and the
NLP team's attention trace. This notebook does not train a substitute model.


In [ ]:
from src.core.io import read_predictions
from src.core.paths import predictions_path
model = "random" if synthetic else "gat_lstm"
if predictions_path(model).exists():
    predictions = read_predictions(model)
    print("Saved artifact:", model, "SYNTHETIC EXAMPLE" if synthetic else "requires provenance review")
    print(predictions.head().to_string(index=False))
else:
    print("Pending: trained GAT/LSTM predictions, checkpoint and attention trace")
